In [ ]:
#machine_learning
#SVM
#KNN
#LogisticRegression
#RandomForest
#DecisionTree

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, plot_confusion_matrix
from sklearn.metrics import precision_recall_curve, plot_precision_recall_curve, plot_roc_curve
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics
import matplotlib.image as pltimg
%matplotlib inline

# Import and print the dataset

In [ ]:
train = pd.read_csv('../input/titanic/train.csv')

In [ ]:
train.head()

# Exploratory Data Analysis

In [ ]:
train.shape

In [ ]:
train.describe()

In [ ]:
train.drop(['PassengerId', 'Name', 'Ticket', 'Fare',"Embarked"], axis = 1, inplace = True)
train.loc[train['Sex']=='male','Sex'] = 1
train.loc[train['Sex']=='female', 'Sex'] = 0

In [ ]:
train['Survived'].value_counts()

In [ ]:
sns.countplot(data=train, x='Survived')

In [ ]:
train.corr()['Survived'].sort_values()

In [ ]:
sns.scatterplot(data=train, x='Age', y='Sex',hue='Survived')

In [ ]:
sns.pairplot(train,diag_kind='kde')

In [ ]:
sns.heatmap(train.corr(), annot=True,cmap='Greens')

In [ ]:
sns.boxplot(data=train, x='Survived', y='Age')

# Missing Values

In [ ]:
train.isnull().sum()

In [ ]:
100*(train.isnull().sum()/len(train))

In [ ]:
def missing_percent(train):
    nan_percent= 100*(train.isnull().sum()/len(train))
    nan_percent= nan_percent[nan_percent>0].sort_values()
    return nan_percent

In [ ]:
nan_percent= missing_percent(train)

In [ ]:
nan_percent

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(x=nan_percent.index, y=nan_percent)
plt.xticks(rotation=90)

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(x=nan_percent.index, y=nan_percent)
plt.xticks(rotation=90)

#Set 1% threshold:
plt.ylim(0,1)

In [ ]:
nan_percent[nan_percent<1]

In [ ]:
train.drop("Cabin", axis = 1, inplace = True)

In [ ]:
train["Age"].fillna(train["Age"].mean(), inplace = True)

In [ ]:
train.isnull().sum()

In [ ]:
# Split Data into train,test

In [ ]:
X = train.drop("Survived", axis = 1, inplace = False)
y = train["Survived"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=101)

# Normalization Data

In [ ]:
scaler=StandardScaler()
scaler.fit(X_train)

In [ ]:
scaler_train=scaler.transform(X_train)
scaler_test=scaler.transform(X_test)

# Fit and test all machin learning models

# 1- Logistic Regression

In [ ]:
Lr_model = LogisticRegression()
Lr_model.fit(scaler_train,y_train)

In [ ]:
y_pred= Lr_model.predict(scaler_test)

In [ ]:
plot_confusion_matrix(Lr_model, scaler_test, y_test)

In [ ]:
print(classification_report(y_test, y_pred))

# 2- KNN

In [ ]:
test_error_rate=[]
for k in range(1,30):
    knn_midel=KNeighborsClassifier(n_neighbors=k)
    knn_midel.fit(scaler_train,y_train)
    y_p_test=knn_midel.predict(scaler_test)
    test_error=1-accuracy_score(y_test,y_p_test)
    test_error_rate.append(test_error)

In [ ]:
test_error_rate

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(range(1,30),test_error_rate,label='test_error')
plt.legend()
plt.xlabel('k Value')
plt.ylabel('Error')

In [ ]:
knn_model=KNeighborsClassifier(n_neighbors=4)
knn_model.fit(scaler_train,y_train)

In [ ]:
y_pred=knn_model.predict(scaler_test)

In [ ]:
plot_confusion_matrix(Lr_model, scaler_test, y_test)

In [ ]:
print(classification_report(y_test, y_pred))

# 3- SVM

In [ ]:
svc = SVC()
svc.fit(scaler_train, y_train)
y_pred = svc.predict(scaler_test)

In [ ]:
plot_confusion_matrix(Lr_model, scaler_test, y_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
kernels = ['Polynomial', 'RBF', 'Sigmoid','Linear']
#A function which returns the corresponding SVC model
def getClassifier(ktype):
    if ktype == 0:
        # Polynomial kernal
        return SVC(kernel='poly', degree=8, gamma="auto")
    elif ktype == 1:
        # Radial Basis Function kernal
        return SVC(kernel='rbf', gamma="auto")
    elif ktype == 2:
        # Sigmoid kernal
        return SVC(kernel='sigmoid', gamma="auto")
    elif ktype == 3:
        # Linear kernal
        return SVC(kernel='linear', gamma="auto")

In [ ]:
kernels = ['Polynomial', 'RBF', 'Sigmoid','Linear']

In [ ]:
for i in range(4):
    svclassifier = getClassifier(i) 
    svclassifier.fit(scaler_train, y_train)# Make prediction
    y_pred = svclassifier.predict(scaler_test)# Evaluate our model
    print("Evaluation:", kernels[i], "kernel")
    print(classification_report(y_test,y_pred))

# 4- Tree

In [ ]:
def train_using_gini(scaler_train,y_train):
  
    # Creating the classifier object
    clf_gini = DecisionTreeClassifier(criterion = "gini",
            random_state = 100,max_depth=3, min_samples_leaf=5)
  
    # Performing training
    clf_gini.fit(scaler_train, y_train)
    return clf_gini

In [ ]:
def tarin_using_entropy(scaler_train, y_train):
  
    # Decision tree with entropy
    clf_entropy = DecisionTreeClassifier(
            criterion = "entropy", random_state = 100,
            max_depth = 3, min_samples_leaf = 5)
  
    # Performing training
    clf_entropy.fit(scaler_train, y_train)
    return clf_entropy

In [ ]:
def prediction(scaler_test, clf_object):
  
    # Predicton on test with giniIndex
    y_pred = clf_object.predict(scaler_test)
    print("Predicted values:")
    print(y_pred)
    return y_pred

In [ ]:
def cal_accuracy(y_test, y_pred):
      
    print("Confusion Matrix: ",
        confusion_matrix(y_test, y_pred))
      
    print ("Accuracy : ",
    accuracy_score(y_test,y_pred)*100)
      
    print("Report : ",
    classification_report(y_test, y_pred))

In [ ]:
    clf_gini = train_using_gini(scaler_train, y_train)
    clf_entropy = tarin_using_entropy(scaler_train, y_train)
      
    # Operational Phase
    print("Results Using Gini Index:")
      
    # Prediction using gini
    y_pred_gini = prediction(X_test, clf_gini)
    cal_accuracy(y_test, y_pred_gini)
      
    print("Results Using Entropy:")
    # Prediction using entropy
    y_pred_entropy = prediction(scaler_test, clf_entropy)
    cal_accuracy(y_test, y_pred_entropy)

# 5- Random Forest

In [ ]:
RF = RandomForestClassifier(n_estimators = 100) 

In [ ]:
RF.fit(scaler_train, y_train)

In [ ]:
plot_confusion_matrix(Lr_model, scaler_test, y_test)

In [ ]:
print(classification_report(y_test, y_pred))

# 6- Ensemble VotingClassifier

In [ ]:
# group / ensemble of models
estimator = []
estimator.append(('LR', 
                  LogisticRegression(solver ='lbfgs', 
                                     multi_class ='multinomial', 
                                     max_iter = 200)))
estimator.append(('SVC', SVC(gamma ='auto', probability = True)))
estimator.append(('DTC', DecisionTreeClassifier()))
  
# Voting Classifier with hard voting
vot_hard = VotingClassifier(estimators = estimator, voting ='hard')
vot_hard.fit(scaler_train, y_train)
y_pred = vot_hard.predict(scaler_test)
  
# using accuracy_score metric to predict accuracy
plot_confusion_matrix(Lr_model, scaler_test, y_test)
print(classification_report(y_test, y_pred))
print("Hard Voting")
  


In [ ]:
# Voting Classifier with soft voting
vot_soft = VotingClassifier(estimators = estimator, voting ='soft')
vot_soft.fit(scaler_train, y_train)
y_pred = vot_soft.predict(scaler_test)
  
# using accuracy_score
plot_confusion_matrix(Lr_model, scaler_test, y_test)
print(classification_report(y_test, y_pred))
print("Soft Voting ")

# Result

In [ ]:
result=pd.DataFrame(data=[79,82,80,79,77,83],index=['Logistic Regression','KNN','SVM','Tree','Random Forest','Ensemble VotingClassifier'],columns=['Accuracy'])

In [ ]:
result

# Ensemble VotingClassifier has the best accuracy